In [1]:
import re

import pandas as pd
import requests

URL = "https://courselistings.wpi.edu/assets/prod-data.json"
# splits "CS 1101 - Introduction to Program Design" into code + title
TITLE_RE = re.compile(r"^(?P<code>[A-Za-z]+ ?\d+[A-Za-z]*) - (?P<title>.*)$")
HTML_TAG_RE = re.compile(r"<[^>]+>")

In [2]:
# pull raw course section entries (one per offered section, not per unique course)
entries = requests.get(URL, timeout=30).json()["Report_Entry"]
print(f"Pulled {len(entries)} course section entries from WPI course listings")

Pulled 3768 course section entries from WPI course listings


In [ ]:
rows = []
for e in entries:
    m = TITLE_RE.match(e["Course_Title"])
    credits = float(e["Credits"])
    level = e["Academic_Level"]
    # grad credits are worth 3/2x their equivalent in UG credits (e.g. 3 grad credits = 4.5 UG credits)
    if level == "Graduate":
        credits_ug, credits_graduate = credits * 3 / 2, credits
    else:
        credits_ug, credits_graduate = credits, credits * 2 / 3
    rows.append(
        {
            "course_code": m["code"],
            "title": m["title"],
            "description": HTML_TAG_RE.sub("", e["Course_Description"]),
            "subject": e["Subject"],
            "credits_raw": credits,
            "credits_ug": credits_ug,
            "credits_graduate": credits_graduate,
            "academic_level": level,
            "department": e["Course_Section_Owner"],
            "start_date": e["Course_Section_Start_Date"],
        }
    )

courses_df = pd.DataFrame(rows).sort_values("start_date")
# collapse sections down to one row per course, keeping the most recent offering
courses_df = courses_df.drop_duplicates("course_code", keep="last")
courses_df = courses_df.set_index("course_code")

print(len(courses_df), "unique courses")
courses_df.head()

In [4]:
# map each department to the list of course codes it owns
courses_by_dept = courses_df.groupby("department").apply(
    lambda g: list(g.index), include_groups=False
)
departments_df = pd.DataFrame({"course_codes": courses_by_dept})

print(len(departments_df), "departments")
departments_df.head()

30 departments


,course_codes
department,
Aerospace Engineering Department,"[AE 601, AE 5233, AE 4220, AE 4210, AE 3310, A..."
Air Force Aerospace Studies (AFROTC) Department,"[AS 2001, AS 1001, AS 4001, AS 3001, AS 1002, ..."
Bioinformatics and Computational Biology Program,"[CS 583, BCB 503, BCB 555, BB 581, BCB 501, BC..."
Biology and Biotechnology Department,"[BB 575, BB 552, BB 504, BB 560, BB 2815, BB 3..."
Biomedical Engineering Department,"[ME 4814, ECE 4023, BME 4023, BME 4813, BME 59..."


In [5]:
courses_df.to_csv("courses.csv")
departments_df.to_csv("departments.csv")